# QPEI — Rescue & Analysis Notebook
### Fixes the school-ID bug in `QPEI_Analysis_Colab.ipynb` and re-runs the full pipeline

**What was wrong:** `resolve_school_id()` in the original notebook promised (in its own docstring) to fall back to the enumerator's assigned school-range whenever a `school_id` cell was malformed, but it never actually did that — it just returned the raw garbage string (Excel drag-fill artifacts like `S21-S24`, `S05-S08`, `S09-`) and flagged it. Nothing downstream dropped the flagged rows either, so every distinct garbage string became its own fake "school." That inflated `n_schools` from the real 32 up to 73 in the exported `qpei_results.json`.

**What this notebook does differently (Section 3):**
1. Cleans well-formed IDs exactly as before (`S1` → `S01`).
2. For anything malformed/a pasted range, looks up the row's `enumerator_id` in the `Id_Enumerator & School` sheet and resolves it to that enumerator's assigned school **only if it's unambiguous** (the enumerator covers exactly one school).
3. Anything still unresolved is flagged **and excluded** from aggregation (the original notebook computed flags but never filtered on them).
4. A rescue log prints exactly which fake IDs the old logic produced and how many rows were dropped per sheet by the fix, so the correction is auditable.
5. Section 8 also attaches each school's real name (pulled straight from the `school_name` column in the response sheets) to every output table, per the Codebook / `Id_Enumerator & School` sheet.

Everything else (reliability, ICC/r_wg, EFA parallel analysis, VIF/HTMT, EWM/CRITIC weighting, TOPSIS, Monte Carlo sensitivity, figures, JSON export) is unchanged from the original pipeline logic.

## 1. Setup

In [ ]:
!pip -q install factor_analyzer pingouin statsmodels openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import json, os, io, re, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Georgia", "serif"],
    "axes.titlesize": 12,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.edgecolor": "#333333",
    "axes.grid": True,
    "grid.alpha": 0.25,
})

FIG_DIR = Path("/content/qpei_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = {"figures": [], "tables": {}, "stats": {}}


## 2. Upload the master data file

In [ ]:
DATA_PATH = None  # set a Drive path here to skip the upload prompt, e.g. "/content/drive/MyDrive/QPEI/master.xlsx"

if DATA_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        # not running in Colab — point this at the file on disk
        DATA_PATH = "Amena_Fazal_Quality_Primary_Education_Master_Google_Sheet_Template.xlsx"

print("Using:", DATA_PATH)


## 3. Indicator–dimension map (35 indicators)

Unchanged from the original — everything downstream reads from this dict.

In [ ]:
DIMENSIONS = {
    "D1_Teacher_Competence": {
        "teacher":  ["tq6", "tq7", "tq8", "tq15"],
        "student":  ["sq1"],
        "observation": ["co19", "co21"],
    },
    "D2_Curriculum_Assessment": {
        "teacher": ["tq3", "tq5", "tq19", "tq20"],
        "observation": ["co27"],
    },
    "D3_Learning_Environment": {
        "environment": ["se2", "se3", "se5", "se6", "se7", "se8", "se9"],
    },
    "D4_Student_Learning": {
        "student": ["sq4", "sq5", "sq6"],
        "observation": ["co9", "co10", "co16"],
    },
    "D5_Leadership_Community": {
        "teacher": ["tq17", "tq18"],
        "parent": ["pq10"],
    },
    "D6_Equity_Inclusion": {
        "teacher": ["tq9"],
        "student": ["sq13", "sq15"],
        "observation": ["co23", "co24", "co25"],
        "environment": ["se4"],
    },
}

BASELINE_WEIGHTS = {
    "D1_Teacher_Competence": 0.20,
    "D2_Curriculum_Assessment": 0.15,
    "D3_Learning_Environment": 0.15,
    "D4_Student_Learning": 0.20,
    "D5_Leadership_Community": 0.15,
    "D6_Equity_Inclusion": 0.15,
}

MISSING_CODES = [99]      # missing/unclear -> excluded from means, tracked
NA_CODES = [88]           # not applicable -> excluded, tracked separately

SHEET_MAP = {
    "teacher": "Teacher_Survey",
    "student": "Student_Questionnaire",
    "parent": "Parent_Survey",
    "observation": "Classroom_Observation",
    "environment": "School_Environment",
}


## 4. Load raw sheets + enumerator→school lookup

In [ ]:
xls = pd.ExcelFile(DATA_PATH)
sheet_names = xls.sheet_names
print(sheet_names)

lookup_sheet = "Id_Enumerator & School" if "Id_Enumerator & School" in sheet_names else "Id_Enumerator_School"
raw = {k: pd.read_excel(DATA_PATH, sheet_name=v, header=0) for k, v in SHEET_MAP.items()}
lookup = pd.read_excel(DATA_PATH, sheet_name=lookup_sheet)
lookup.columns = [str(c).strip().lower() for c in lookup.columns]
lookup["enumerator_id"] = lookup["enumerator_id"].astype(str).str.strip().str.upper()
print(lookup.head(10))


## 5. RESCUE: clean & resolve `school_id` (the actual fix)

`resolve_school_id_FIXED()` replaces the original `resolve_school_id()`. It (a) cleans well-formed IDs, (b) for malformed/range values, expands the matched enumerator's assigned range from `Id_Enumerator & School` and resolves *only* if that enumerator covers exactly one school, and (c) flags everything else for exclusion. We keep both the old buggy result and the new fixed result side-by-side so the rescue is auditable, then print a log of exactly what got dropped/fixed.

In [ ]:
VALID_SCHOOL_RE = re.compile(r"^S([0-3][0-9])$")

def expand_range(r):
    """Expand an enumerator's assigned range e.g. 'S01-S04' -> ['S01','S02','S03','S04']."""
    r = str(r).strip().upper()
    m = re.match(r"^S0*([0-9]+)-S0*([0-9]+)$", r)
    if not m:
        return []
    lo, hi = int(m.group(1)), int(m.group(2))
    return [f"S{n:02d}" for n in range(lo, hi + 1)]

enum_to_schools = {row["enumerator_id"]: expand_range(row["school_id"]) for _, row in lookup.iterrows()}

def clean_school_id(x):
    """Standardize a raw school_id value: uppercase, strip, fix common artifacts."""
    if pd.isna(x):
        return None
    s = str(x).strip().upper().replace(" ", "")
    bn = "০১২৩৪৫৬৭৮৯"   # Bengali digit -> ASCII digit map
    for i, d in enumerate(bn):
        s = s.replace(d, str(i))
    m = re.match(r"^S0*([0-9]{1,2})$", s)
    if m:
        return f"S{int(m.group(1)):02d}"
    return s  # leave range-artifacts (e.g. S21-S24) as-is; resolved (or dropped) below

def resolve_school_id_FIXED(raw_school_id, enumerator_id, enum_to_schools):
    """Actually implements what the original docstring promised."""
    cleaned = clean_school_id(raw_school_id)
    if cleaned:
        m = VALID_SCHOOL_RE.match(cleaned)
        if m and 1 <= int(m.group(1)) <= 32:
            return cleaned, False  # already a valid single school id
    enum_key = str(enumerator_id).strip().upper() if pd.notna(enumerator_id) else None
    candidates = enum_to_schools.get(enum_key, [])
    if len(candidates) == 1:
        return candidates[0], False       # unambiguous enumerator fallback resolved it
    return cleaned, True                   # still unresolvable -> flag for exclusion

rescue_log = []
fake_ids_found = set()
for src, df in raw.items():
    df.columns = [str(c).strip() for c in df.columns]
    if "school_id" not in df.columns:
        continue
    enum_col = "enumerator_id" if "enumerator_id" in df.columns else None

    old_ids, old_flag, new_ids, new_flag = [], [], [], []
    for _, r in df.iterrows():
        sid_raw = r.get("school_id")
        enum_id = r.get(enum_col) if enum_col else None

        c = clean_school_id(sid_raw)
        m = VALID_SCHOOL_RE.match(c) if c else None
        old_valid = bool(m and 1 <= int(m.group(1)) <= 32)
        old_ids.append(c); old_flag.append(not old_valid)
        if not old_valid and c is not None:
            fake_ids_found.add(str(c))

        nid, nflag = resolve_school_id_FIXED(sid_raw, enum_id, enum_to_schools)
        new_ids.append(nid); new_flag.append(nflag)

    df["school_id_clean"] = new_ids
    df["school_id_flagged"] = new_flag
    raw[src] = df

    rescue_log.append({
        "sheet": src,
        "n_rows": len(df),
        "n_flagged_school_id_OLD_BUGGY": int(sum(old_flag)),
        "n_rows_dropped_after_fix": int(sum(new_flag)),
        "n_real_schools_recovered": len(set(i for i, f in zip(new_ids, new_flag) if not f)),
    })

rescue_log_df = pd.DataFrame(rescue_log)
print("Fake school-id strings the OLD logic would have kept as their own 'schools':")
print(sorted(str(x) for x in fake_ids_found))
print()
print(rescue_log_df)
RESULTS["tables"]["id_rescue_log"] = rescue_log_df.to_dict(orient="records")


## 6. Recode missing/NA sentinel values across all item columns

In [ ]:
def recode_missing(df, item_cols):
    miss_report = []
    for c in item_cols:
        if c not in df.columns:
            continue
        col = df[c]
        n_miss = col.isin(MISSING_CODES).sum() + col.isna().sum()
        n_na = col.isin(NA_CODES).sum()
        n_total = len(col)
        miss_report.append({"item": c, "n_missing": int(n_miss), "n_not_applicable": int(n_na),
                             "pct_missing": round(100 * n_miss / n_total, 2)})
        df[c] = col.where(~col.isin(MISSING_CODES + NA_CODES), np.nan)
    return pd.DataFrame(miss_report)

all_items_by_source = {}
for dim, srcmap in DIMENSIONS.items():
    for src, items in srcmap.items():
        all_items_by_source.setdefault(src, set()).update(items)

missingness_tables = {}
for src, df in raw.items():
    items = sorted(all_items_by_source.get(src, []))
    if not items:
        continue
    missingness_tables[src] = recode_missing(df, items)

for src, tbl in missingness_tables.items():
    print("===", src, "===")
    print(tbl)
RESULTS["tables"]["missingness"] = {k: v.to_dict(orient="records") for k, v in missingness_tables.items()}


## 7. Reliability at respondent level (before aggregation)

Cronbach's alpha *and* McDonald's omega, computed per dimension × source item battery (unchanged).

In [ ]:
import pingouin as pg
from factor_analyzer import FactorAnalyzer

def mcdonalds_omega(df_items):
    """Single-factor omega from EFA loadings: omega = (sum(loadings))^2 /
    [(sum(loadings))^2 + sum(1 - loadings^2)]"""
    d = df_items.dropna()
    if d.shape[0] < 10 or d.shape[1] < 2:
        return np.nan
    try:
        fa = FactorAnalyzer(n_factors=1, rotation=None, method="ml")
        fa.fit(d)
        loadings = fa.loadings_.flatten()
        num = loadings.sum() ** 2
        den = num + np.sum(1 - loadings ** 2)
        return float(num / den) if den > 0 else np.nan
    except Exception:
        return np.nan

reliability_rows = []
for dim, srcmap in DIMENSIONS.items():
    for src, items in srcmap.items():
        if len(items) < 2:
            continue
        df = raw[src]
        cols = [c for c in items if c in df.columns]
        sub = df[cols].apply(pd.to_numeric, errors="coerce")
        try:
            alpha = pg.cronbach_alpha(data=sub.dropna())[0]
        except Exception:
            alpha = np.nan
        omega = mcdonalds_omega(sub)
        reliability_rows.append({
            "dimension": dim, "source": src, "n_items": len(cols),
            "n_respondents": int(sub.dropna().shape[0]),
            "cronbach_alpha": round(alpha, 3) if pd.notna(alpha) else None,
            "mcdonald_omega": round(omega, 3) if pd.notna(omega) else None,
        })

reliability_df = pd.DataFrame(reliability_rows)
print(reliability_df)
RESULTS["tables"]["reliability_alpha_omega"] = reliability_df.to_dict(orient="records")


## 8. Aggregation justification: ICC(1), ICC(2), r_wg

Justifies collapsing respondent-level items to school level (unchanged, but now grouped on the **rescued** `school_id_clean` with flagged rows excluded).

In [ ]:
def rwg_uniform(x, n_scale_points=5):
    """James, Demaree & Wolf (1984) r_wg with uniform null distribution."""
    x = pd.Series(x).dropna()
    if len(x) < 2:
        return np.nan
    var_obs = x.var(ddof=1)
    var_exp = (n_scale_points ** 2 - 1) / 12
    return max(0.0, 1 - (var_obs / var_exp)) if var_exp > 0 else np.nan

def icc1_icc2(df, item_col, group_col="school_id_clean"):
    d = df[~df["school_id_flagged"]][[group_col, item_col]].dropna()
    d[item_col] = pd.to_numeric(d[item_col], errors="coerce")
    d = d.dropna()
    if d[group_col].nunique() < 2:
        return np.nan, np.nan
    groups = d.groupby(group_col)[item_col]
    k_bar = groups.count().mean()
    aov = pg.anova(data=d, dv=item_col, between=group_col, detailed=True)
    try:
        ms_between = aov.loc[aov["Source"] == group_col, "MS"].values[0]
        ms_within = aov.loc[aov["Source"] == "Within", "MS"].values[0]
    except Exception:
        return np.nan, np.nan
    icc1 = (ms_between - ms_within) / (ms_between + (k_bar - 1) * ms_within)
    icc2 = (ms_between - ms_within) / ms_between if ms_between > 0 else np.nan
    return icc1, icc2

icc_rows = []
for dim, srcmap in DIMENSIONS.items():
    for src, items in srcmap.items():
        if src not in ("teacher", "student", "parent"):
            continue
        df = raw[src]
        if "school_id_clean" not in df.columns:
            continue
        for item in items:
            if item not in df.columns:
                continue
            icc1, icc2 = icc1_icc2(df, item)
            rwg_vals = df[~df["school_id_flagged"]].groupby("school_id_clean")[item].apply(lambda s: rwg_uniform(s))
            icc_rows.append({
                "dimension": dim, "source": src, "item": item,
                "ICC1": round(icc1, 3) if pd.notna(icc1) else None,
                "ICC2": round(icc2, 3) if pd.notna(icc2) else None,
                "mean_rwg": round(rwg_vals.mean(), 3) if len(rwg_vals) else None,
            })

icc_df = pd.DataFrame(icc_rows)
print(icc_df)
RESULTS["tables"]["aggregation_justification_icc_rwg"] = icc_df.to_dict(orient="records")


## 9. Exploratory Factor Analysis — Horn's Parallel Analysis (not Kaiser)

Unchanged.

In [ ]:
def parallel_analysis(data, n_iter=1000, percentile=95, seed=42):
    rng = np.random.default_rng(seed)
    d = data.dropna()
    n, p = d.shape
    if n < 10 or p < 3:
        return None
    corr = d.corr().values
    obs_eigs = np.linalg.eigvalsh(corr)[::-1]
    sim_eigs = np.zeros((n_iter, p))
    for i in range(n_iter):
        sim = rng.standard_normal((n, p))
        sim_corr = np.corrcoef(sim, rowvar=False)
        sim_eigs[i] = np.linalg.eigvalsh(sim_corr)[::-1]
    threshold = np.percentile(sim_eigs, percentile, axis=0)
    n_factors = int(np.sum(obs_eigs > threshold))
    return {"observed_eigenvalues": obs_eigs.tolist(), "pa_threshold_eigenvalues": threshold.tolist(),
            "n_factors_retained": n_factors, "n_items": p, "n_respondents": n}

efa_results = {}
for dim, srcmap in DIMENSIONS.items():
    for src, items in srcmap.items():
        if len(items) < 3:
            continue
        df = raw[src]
        cols = [c for c in items if c in df.columns]
        sub = df[cols].apply(pd.to_numeric, errors="coerce")
        pa = parallel_analysis(sub)
        if pa is None:
            continue
        key = f"{dim}__{src}"
        efa_results[key] = pa
        k = max(1, pa["n_factors_retained"])
        try:
            fa = FactorAnalyzer(n_factors=k, rotation="promax" if k > 1 else None, method="ml")
            fa.fit(sub.dropna())
            efa_results[key]["loadings"] = fa.loadings_.round(3).tolist()
            efa_results[key]["items"] = cols
        except Exception as e:
            efa_results[key]["loadings_error"] = str(e)

RESULTS["stats"]["efa_parallel_analysis"] = efa_results
print(f"EFA run on {len(efa_results)} dimension x source item batteries with >=3 items.")


## 10. Formative-model diagnostics: multicollinearity (VIF)

Unchanged.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(df_items):
    d = df_items.dropna()
    if d.shape[0] < 10 or d.shape[1] < 2:
        return pd.DataFrame()
    d = d.assign(const=1)
    vifs = [variance_inflation_factor(d.values, i) for i in range(d.shape[1] - 1)]
    return pd.DataFrame({"item": d.columns[:-1], "VIF": [round(v, 2) for v in vifs]})

vif_rows = []
for dim, srcmap in DIMENSIONS.items():
    for src, items in srcmap.items():
        if len(items) < 2:
            continue
        df = raw[src]
        cols = [c for c in items if c in df.columns]
        sub = df[cols].apply(pd.to_numeric, errors="coerce")
        vif_tbl = compute_vif(sub)
        for _, r in vif_tbl.iterrows():
            vif_rows.append({"dimension": dim, "source": src, "item": r["item"], "VIF": r["VIF"],
                              "flag_high_collinearity": bool(r["VIF"] > 3.0)})

vif_df = pd.DataFrame(vif_rows)
print(vif_df)
RESULTS["tables"]["formative_vif"] = vif_df.to_dict(orient="records")


## 11. School-level aggregation and 0–100 normalization

**This is the second half of the rescue**: the original `aggregate_to_school()` grouped by `school_id_clean` for every row, flagged or not. Here we explicitly drop flagged (unresolvable) rows before grouping, so only real, unambiguous schools make it into `qpei_df`.

In [ ]:
def aggregate_to_school(df, items, group_col="school_id_clean"):
    cols = [c for c in items if c in df.columns]
    valid = df[df[group_col].notna() & ~df["school_id_flagged"]]
    sub = valid[[group_col] + cols].copy()
    for c in cols:
        sub[c] = pd.to_numeric(sub[c], errors="coerce")
    return sub.groupby(group_col)[cols].mean()

school_level = {}
for src, df in raw.items():
    if "school_id_clean" not in df.columns:
        continue
    items = sorted(all_items_by_source.get(src, []))
    if not items:
        continue
    school_level[src] = aggregate_to_school(df, items)

all_schools = sorted(set().union(*[s.index for s in school_level.values()]))
qpei_df = pd.DataFrame(index=all_schools)
qpei_df.index.name = "school_id"

for src, tbl in school_level.items():
    tbl = tbl.reindex(all_schools)
    for col in tbl.columns:
        qpei_df[col] = 25 * (tbl[col] - 1)   # 0-100 normalization

print("Rescued school count:", qpei_df.shape[0], "(vs. 73 fake entries the buggy pipeline produced)")
qpei_df.head()


## 12. Match school_id → real school name (Codebook / `Id_Enumerator & School`)

Pulls the actual `school_name` recorded in each response sheet for every rescued school_id, so every downstream table can be read by name, not just code.

In [ ]:
name_map = {}
meta_map = {}
for src, df in raw.items():
    if "school_id_clean" not in df.columns or "school_name" not in df.columns:
        continue
    for _, r in df[~df["school_id_flagged"]].iterrows():
        sid, name = r["school_id_clean"], r.get("school_name")
        if pd.isna(name) or str(name).strip() in ("", "School name"):
            continue
        name_map.setdefault(sid, str(name).strip())
        meta_map.setdefault(sid, {
            "district": r.get("district"),
            "upazila": r.get("upazila"),
            "division": r.get("division"),
        })

qpei_df["school_name"] = qpei_df.index.map(name_map).fillna("(not entered in master sheet)")
qpei_df["district"] = qpei_df.index.map(lambda s: meta_map.get(s, {}).get("district"))
qpei_df["upazila"] = qpei_df.index.map(lambda s: meta_map.get(s, {}).get("upazila"))
qpei_df["division"] = qpei_df.index.map(lambda s: meta_map.get(s, {}).get("division"))
print(qpei_df[["school_name", "district", "upazila", "division"]])


## 13. Domain scores, QPEI (baseline weights), and dimension-level HTMT

Unchanged.

In [ ]:
for dim, srcmap in DIMENSIONS.items():
    cols = [c for src, items in srcmap.items() for c in items if c in qpei_df.columns]
    qpei_df[dim] = qpei_df[cols].mean(axis=1)

domain_cols = list(DIMENSIONS.keys())
qpei_df["QPEI_baseline"] = sum(qpei_df[d] * w for d, w in BASELINE_WEIGHTS.items())

domain_corr = qpei_df[domain_cols].corr(method="pearson")
print(domain_corr.round(2))

def htmt(df, dims):
    """Simplified HTMT proxy at domain-score level."""
    out = {}
    for i, d1 in enumerate(dims):
        for d2 in dims[i+1:]:
            r_between = df[[d1, d2]].corr().iloc[0, 1]
            out[f"{d1} vs {d2}"] = round(abs(r_between), 3)
    return out

htmt_domain = htmt(qpei_df, domain_cols)
RESULTS["tables"]["domain_correlation_matrix"] = domain_corr.round(3).to_dict()
RESULTS["tables"]["htmt_domain_level"] = htmt_domain
print(htmt_domain)


## 14. Alternative weighting: Entropy Weight Method (EWM) and CRITIC

Unchanged.

In [ ]:
def entropy_weights(df, cols):
    X = df[cols].clip(lower=0.0001)
    P = X.div(X.sum(axis=0), axis=1)
    k = 1 / np.log(len(P))
    E = -k * (P * np.log(P)).sum(axis=0)
    d = 1 - E
    return (d / d.sum()).to_dict()

def critic_weights(df, cols):
    X = df[cols]
    std = X.std()
    corr = X.corr()
    conflict = (1 - corr).sum()
    C = std * conflict
    return (C / C.sum()).to_dict()

ewm_w = entropy_weights(qpei_df, domain_cols)
critic_w = critic_weights(qpei_df, domain_cols)

weights_compare = pd.DataFrame({
    "baseline_theory": BASELINE_WEIGHTS,
    "entropy_EWM": ewm_w,
    "CRITIC": critic_w,
}).round(3)
print(weights_compare)
RESULTS["tables"]["weighting_schemes"] = weights_compare.reset_index().rename(columns={"index": "dimension"}).to_dict(orient="records")

qpei_df["QPEI_EWM"] = sum(qpei_df[d] * ewm_w[d] for d in domain_cols)
qpei_df["QPEI_CRITIC"] = sum(qpei_df[d] * critic_w[d] for d in domain_cols)


## 15. Decision-making cross-check: TOPSIS ranking

Unchanged.

In [ ]:
def topsis(df, cols, weights):
    X = df[cols].values.astype(float)
    norm = X / np.sqrt((X ** 2).sum(axis=0))
    w = np.array([weights[c] for c in cols])
    weighted = norm * w
    ideal_best = weighted.max(axis=0)
    ideal_worst = weighted.min(axis=0)
    dist_best = np.sqrt(((weighted - ideal_best) ** 2).sum(axis=1))
    dist_worst = np.sqrt(((weighted - ideal_worst) ** 2).sum(axis=1))
    score = dist_worst / (dist_best + dist_worst)
    return score

qpei_df["TOPSIS_score"] = topsis(qpei_df, domain_cols, BASELINE_WEIGHTS)
qpei_df["rank_QPEI_baseline"] = qpei_df["QPEI_baseline"].rank(ascending=False)
qpei_df["rank_TOPSIS"] = qpei_df["TOPSIS_score"].rank(ascending=False)

from scipy.stats import spearmanr
rho_topsis, p_topsis = spearmanr(qpei_df["rank_QPEI_baseline"], qpei_df["rank_TOPSIS"])
print(f"Spearman rho (QPEI baseline rank vs TOPSIS rank): {rho_topsis:.3f} (p={p_topsis:.4f})")
RESULTS["stats"]["topsis_vs_qpei_baseline_spearman_rho"] = round(rho_topsis, 3)


## 16. Monte Carlo sensitivity analysis

Unchanged. Perturbs each baseline dimension weight ±25% for 10,000 iterations and tracks rank volatility.

In [ ]:
def monte_carlo_sensitivity(df, dims, base_weights, n_iter=10000, perturb=0.25, seed=1):
    rng = np.random.default_rng(seed)
    base = np.array([base_weights[d] for d in dims])
    X = df[dims].values
    rank_matrix = np.zeros((n_iter, len(df)))
    for i in range(n_iter):
        noise = rng.uniform(1 - perturb, 1 + perturb, size=len(dims))
        w = base * noise
        w = w / w.sum()
        scores = X @ w
        rank_matrix[i] = pd.Series(scores).rank(ascending=False).values
    return rank_matrix

rank_matrix = monte_carlo_sensitivity(qpei_df, domain_cols, BASELINE_WEIGHTS)
rank_summary = pd.DataFrame({
    "school_id": qpei_df.index,
    "school_name": qpei_df["school_name"].values,
    "baseline_rank": qpei_df["rank_QPEI_baseline"].values,
    "mc_rank_mean": rank_matrix.mean(axis=0).round(2),
    "mc_rank_p5": np.percentile(rank_matrix, 5, axis=0),
    "mc_rank_p95": np.percentile(rank_matrix, 95, axis=0),
}).sort_values("baseline_rank")

print(rank_summary.head(10))
RESULTS["tables"]["monte_carlo_rank_volatility"] = rank_summary.round(2).to_dict(orient="records")

rho_spearman_stability = []
base_scores = qpei_df["QPEI_baseline"].values
for i in range(0, 10000, 50):
    noise = np.random.default_rng(i).uniform(0.75, 1.25, size=len(domain_cols))
    w = np.array([BASELINE_WEIGHTS[d] for d in domain_cols]) * noise
    w = w / w.sum()
    alt_scores = qpei_df[domain_cols].values @ w
    rho, _ = spearmanr(base_scores, alt_scores)
    rho_spearman_stability.append(rho)

RESULTS["stats"]["monte_carlo_spearman_rho_mean"] = round(float(np.mean(rho_spearman_stability)), 3)
RESULTS["stats"]["monte_carlo_spearman_rho_min"] = round(float(np.min(rho_spearman_stability)), 3)
print("Mean Spearman rho across perturbations:", RESULTS["stats"]["monte_carlo_spearman_rho_mean"])


## 17. Figures

Same six figures as the original pipeline, serif font applied globally from Section 1. Figure 3 now labels the x-axis with real school names where available.

In [ ]:
# Fig 1 — Parallel analysis scree plot (example battery)
example_key = next((k for k in efa_results if "teacher" in k), list(efa_results.keys())[0])
pa = efa_results[example_key]
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(1, len(pa["observed_eigenvalues"]) + 1)
ax.plot(x, pa["observed_eigenvalues"], marker="o", label="Observed eigenvalues", color="#0C447C")
ax.plot(x, pa["pa_threshold_eigenvalues"], marker="s", linestyle="--", label="Parallel analysis (95th pct.)", color="#BA7517")
ax.axhline(1, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel("Factor"); ax.set_ylabel("Eigenvalue")
ax.set_title(f"Parallel analysis: {example_key.replace('_', ' ')}")
ax.legend()
fname = FIG_DIR / "fig1_parallel_analysis_example.png"
plt.tight_layout(); plt.savefig(fname); plt.show()
RESULTS["figures"].append({"id": "fig1_parallel_analysis", "file": str(fname)})


In [ ]:
# Fig 2 — Domain correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(domain_corr.values, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(domain_cols))); ax.set_xticklabels([d.split("_",1)[0] for d in domain_cols], rotation=45, ha="right")
ax.set_yticks(range(len(domain_cols))); ax.set_yticklabels([d.split("_",1)[0] for d in domain_cols])
for i in range(len(domain_cols)):
    for j in range(len(domain_cols)):
        ax.text(j, i, f"{domain_corr.values[i,j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Inter-dimension correlation matrix (school level)")
fig.colorbar(im, ax=ax, shrink=0.8)
fname = FIG_DIR / "fig2_domain_correlation_heatmap.png"
plt.tight_layout(); plt.savefig(fname); plt.show()
RESULTS["figures"].append({"id": "fig2_domain_correlation", "file": str(fname)})


In [ ]:
# Fig 3 — QPEI scores by school (baseline weights), sorted, labeled by school NAME where known
sorted_df = qpei_df.sort_values("QPEI_baseline", ascending=False)
xlabels = [n if n != "(not entered in master sheet)" else i for i, n in zip(sorted_df.index, sorted_df["school_name"])]
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(sorted_df)), sorted_df["QPEI_baseline"], color="#3B6FA0", edgecolor="#0C447C")
ax.set_xticks(range(len(sorted_df))); ax.set_xticklabels(xlabels, rotation=90, fontsize=7)
ax.set_ylabel("QPEI score (0-100)")
ax.set_title("School-level QPEI scores (baseline weighting)")
ax.axhline(sorted_df["QPEI_baseline"].mean(), color="#BA7517", linestyle="--", linewidth=1, label="Sample mean")
ax.legend()
fname = FIG_DIR / "fig3_qpei_by_school.png"
plt.tight_layout(); plt.savefig(fname); plt.show()
RESULTS["figures"].append({"id": "fig3_qpei_by_school", "file": str(fname)})


In [ ]:
# Fig 4 — Weighting scheme comparison (baseline vs EWM vs CRITIC)
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(domain_cols)); width = 0.26
labels = [d.split("_", 1)[0] for d in domain_cols]
ax.bar(x - width, [BASELINE_WEIGHTS[d] for d in domain_cols], width, label="Theory-informed", color="#0C447C")
ax.bar(x, [ewm_w[d] for d in domain_cols], width, label="Entropy (EWM)", color="#1D9E75")
ax.bar(x + width, [critic_w[d] for d in domain_cols], width, label="CRITIC", color="#BA7517")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("Weight")
ax.set_title("Dimension weights: theory-informed vs. objective schemes")
ax.legend()
fname = FIG_DIR / "fig4_weighting_comparison.png"
plt.tight_layout(); plt.savefig(fname); plt.show()
RESULTS["figures"].append({"id": "fig4_weighting_comparison", "file": str(fname)})


In [ ]:
# Fig 5 — Monte Carlo rank-volatility (school-level boxplot, sorted by baseline rank)
order = qpei_df.sort_values("rank_QPEI_baseline").index
order_idx = [list(qpei_df.index).index(s) for s in order]
order_labels = [n if n != "(not entered in master sheet)" else i for i, n in zip(order, qpei_df.loc[order, "school_name"])]
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([rank_matrix[:, i] for i in order_idx], showfliers=False)
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order_labels, rotation=90, fontsize=7)
ax.set_ylabel("Rank across 10,000 weight perturbations")
ax.invert_yaxis()
ax.set_title("Monte Carlo sensitivity: rank volatility per school (±25% weight perturbation)")
fname = FIG_DIR / "fig5_monte_carlo_rank_volatility.png"
plt.tight_layout(); plt.savefig(fname); plt.show()
RESULTS["figures"].append({"id": "fig5_monte_carlo_volatility", "file": str(fname)})


In [ ]:
# Fig 6 — Radar chart: dimension profile, top vs bottom QPEI quartile
q_hi = qpei_df[qpei_df["QPEI_baseline"] >= qpei_df["QPEI_baseline"].quantile(0.75)][domain_cols].mean()
q_lo = qpei_df[qpei_df["QPEI_baseline"] <= qpei_df["QPEI_baseline"].quantile(0.25)][domain_cols].mean()

labels = [d.split("_", 1)[0] for d in domain_cols]
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for series, name, color in [(q_hi, "Top quartile schools", "#1D9E75"), (q_lo, "Bottom quartile schools", "#BA3B1E")]:
    vals = series.tolist(); vals += vals[:1]
    ax.plot(angles, vals, marker="o", label=name, color=color)
    ax.fill(angles, vals, alpha=0.12, color=color)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, fontsize=8)
ax.set_title("Dimension profile: top vs. bottom QPEI quartile schools")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
fname = FIG_DIR / "fig6_radar_quartile_profile.png"
plt.tight_layout(); plt.savefig(fname); plt.show()
RESULTS["figures"].append({"id": "fig6_radar_profile", "file": str(fname)})


## 18. Export final school-level results table and `qpei_results_RESCUED.json`

Same export shape as the original `qpei_results.json`, plus `school_name`/`district`/`upazila`/`division` columns and the `id_rescue_log` table so the fix is documented inside the output file itself.

In [ ]:
final_cols = ["school_name", "district", "upazila", "division"] + domain_cols + [
    "QPEI_baseline", "QPEI_EWM", "QPEI_CRITIC", "TOPSIS_score", "rank_QPEI_baseline", "rank_TOPSIS"]
final_table = qpei_df[final_cols].round(2).reset_index()
RESULTS["tables"]["school_level_scores"] = final_table.to_dict(orient="records")

RESULTS["config"] = {
    "dimensions": DIMENSIONS,
    "baseline_weights": BASELINE_WEIGHTS,
    "n_schools": len(qpei_df),
    "n_schools_in_original_buggy_export": 73,
}

out_path = "/content/qpei_results_RESCUED.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, indent=2, default=str)

final_table.to_csv("/content/qpei_school_scores_RESCUED.csv", index=False)

print("Saved:", out_path)
print("Saved:", "/content/qpei_school_scores_RESCUED.csv")
print("Figures saved in:", FIG_DIR)
final_table.sort_values("rank_QPEI_baseline")


## 19. Zip and download everything

In [ ]:
import shutil
shutil.make_archive("/content/qpei_outputs_rescued", "zip", root_dir="/content", base_dir="qpei_figures")
import zipfile
with zipfile.ZipFile("/content/qpei_outputs_rescued.zip", "a") as z:
    z.write(out_path, arcname="qpei_results_RESCUED.json")
    z.write("/content/qpei_school_scores_RESCUED.csv", arcname="qpei_school_scores_RESCUED.csv")

try:
    from google.colab import files
    files.download("/content/qpei_outputs_rescued.zip")
except ImportError:
    print("Not in Colab — find the zip at /content/qpei_outputs_rescued.zip")
